# Latent Space Emulation

Instead of emulating each grid point independently, we compress the full
output curve into a low-dimensional latent code and emulate that.

### Approaches
- **PCA**: linear compression, 3 components capture >99.99% of variance
- **Autoencoder**: nonlinear compression via PyTorch encoder/decoder

### Pipeline
```
params → [emu_0(params), ..., emu_K(params)] → latent code → decoder → full curve
```

In [ ]:
import time

import numpy as np
import matplotlib.pyplot as plt

from tissage_cosmique.computations.distances import comoving_angular_distance
from tissage_cosmique.emulators import (
    GPEmulator,
    PCACodec,
    AutoencoderCodec,
    LatentEmulator,
    build_training_data,
    params_to_feature_matrix,
)

In [ ]:
PARAM_NAMES = ["Omega_c", "h", "sigma8"]
FIXED = dict(Omega_b=0.0486, n_s=0.9667, Omega_k=0.0, w0=-1.0, wa=0.0)
a_grid = np.linspace(0.2, 0.8, 30)

rng = np.random.default_rng(42)
n_train, n_test = 50, 10

def make_samples(n, rng):
    return [
        {"Omega_c": rng.uniform(0.22, 0.32), "h": rng.uniform(0.62, 0.75),
         "sigma8": rng.uniform(0.77, 0.87), **FIXED}
        for _ in range(n)
    ]

train_samples = make_samples(n_train, rng)
test_samples = make_samples(n_test, rng)

# Compute all training curves
Y_train = np.array([comoving_angular_distance(p, a_grid) for p in train_samples])
Y_test = np.array([comoving_angular_distance(p, a_grid) for p in test_samples])
print(f"Training curves: {Y_train.shape}  (50 cosmologies x 30 scale factors)")
print(f"Test curves:     {Y_test.shape}")

## 1. PCA analysis: how many components do we need?

In [ ]:
pca_full = PCACodec(n_components=10)
pca_full.fit(Y_train)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.bar(range(1, 11), pca_full.explained_variance_ratio * 100)
ax.set_xlabel("Component")
ax.set_ylabel("Variance explained [%]")
ax.set_title("PCA explained variance")

ax = axes[1]
ax.plot(range(1, 11), pca_full.cumulative_variance, "bo-")
ax.axhline(0.9999, color="red", linestyle="--", label="99.99%")
ax.set_xlabel("Number of components")
ax.set_ylabel("Cumulative variance")
ax.set_title("Cumulative variance")
ax.legend()
ax.set_ylim(0.99, 1.001)

plt.tight_layout()
plt.show()

for k in [1, 2, 3, 5]:
    print(f"  {k} components: {pca_full.cumulative_variance[k-1]:.8f}")

## 2. PCA + GP latent emulator

In [ ]:
%%time
pca_le = LatentEmulator(
    codec=PCACodec(n_components=3),
    emulator_factory=lambda: GPEmulator(feature_names=PARAM_NAMES),
)
pca_le.fit(train_samples, comoving_angular_distance, a_grid, param_names=PARAM_NAMES)

meta = pca_le.metadata
print(f"Codec: {meta['codec']['type']}, {meta['n_latent']} latent dimensions")
print(f"Variance explained: {meta['codec']['total_explained']:.6f}")
print(f"Emulator type: {meta['emulator_type']}")

In [ ]:
# Validate on test set
rel_errors = []
for params in test_samples:
    truth = comoving_angular_distance(params, a_grid)
    pred = pca_le.predict(params)
    rel_err = np.abs(pred - truth) / np.maximum(np.abs(truth), 1.0)
    rel_errors.append(np.mean(rel_err))

print(f"PCA+GP latent emulator — test set:")
print(f"  Mean relative error: {np.mean(rel_errors):.4%}")
print(f"  Max relative error:  {np.max(rel_errors):.4%}")

## 3. Compare: latent emulator vs direct per-point emulator

In [ ]:
# Direct per-point GP for comparison
X_direct, y_direct = build_training_data(
    comoving_angular_distance, train_samples, a_grid, param_names=PARAM_NAMES,
)
direct_emu = GPEmulator(feature_names=PARAM_NAMES + ["a"])
direct_emu.fit(X_direct, y_direct)

# Speed comparison
n_calls = 100
test_p = test_samples[0]

t0 = time.time()
for _ in range(n_calls):
    pca_le.predict(test_p)
latent_time = time.time() - t0

t0 = time.time()
for _ in range(n_calls):
    X = params_to_feature_matrix(test_p, a_grid, param_names=PARAM_NAMES)
    direct_emu.predict(X)
direct_time = time.time() - t0

print(f"Latent (PCA+GP, 3 dims):  {latent_time/n_calls*1000:.1f} ms/call")
print(f"Direct (GP, 30 points):   {direct_time/n_calls*1000:.1f} ms/call")
print(f"Speedup: {direct_time/latent_time:.1f}x")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, params in enumerate(test_samples[:3]):
    truth = comoving_angular_distance(params, a_grid)
    latent_pred = pca_le.predict(params)
    X = params_to_feature_matrix(params, a_grid, param_names=PARAM_NAMES)
    direct_pred = direct_emu.predict(X)

    ax = axes[i]
    ax.plot(a_grid, truth, "k-", lw=2, label="pyccl")
    ax.plot(a_grid, latent_pred, "r--", lw=1.5, label="PCA+GP latent")
    ax.plot(a_grid, direct_pred, "b:", lw=1.5, label="Direct GP")
    ax.set_xlabel("Scale factor a")
    ax.set_ylabel("Distance [Mpc]")
    ax.set_title(f"$\\Omega_c$={params['Omega_c']:.3f}, h={params['h']:.3f}")
    if i == 0:
        ax.legend(fontsize=8)

fig.suptitle("Latent emulator vs direct emulator vs pyccl", fontsize=12)
plt.tight_layout()
plt.show()

## 4. Autoencoder codec

In [ ]:
%%time
ae_le = LatentEmulator(
    codec=AutoencoderCodec(n_latent=3, hidden_layers=[64, 32], n_epochs=500, seed=42),
    emulator_factory=lambda: GPEmulator(feature_names=PARAM_NAMES),
)
ae_le.fit(train_samples, comoving_angular_distance, a_grid, param_names=PARAM_NAMES)

meta = ae_le.metadata
print(f"Codec: {meta['codec']['type']}, {meta['n_latent']} latent dims")
print(f"Reconstruction loss: {meta['codec']['training_loss']:.6f}")

In [ ]:
ae_errors = []
for params in test_samples:
    truth = comoving_angular_distance(params, a_grid)
    pred = ae_le.predict(params)
    rel_err = np.abs(pred - truth) / np.maximum(np.abs(truth), 1.0)
    ae_errors.append(np.mean(rel_err))

print(f"Autoencoder+GP latent emulator — test set:")
print(f"  Mean relative error: {np.mean(ae_errors):.4%}")
print(f"  Max relative error:  {np.max(ae_errors):.4%}")
print(f"\nComparison:")
print(f"  PCA+GP:         {np.mean(rel_errors):.4%}")
print(f"  Autoencoder+GP: {np.mean(ae_errors):.4%}")

## 5. Visualize the latent space

In [ ]:
# Encode all training curves into PCA latent space
Z_pca = pca_le.codec.encode(Y_train)

# Color by Omega_c
omega_c_vals = [p["Omega_c"] for p in train_samples]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
pairs = [(0, 1), (0, 2), (1, 2)]
for ax, (i, j) in zip(axes, pairs):
    sc = ax.scatter(Z_pca[:, i], Z_pca[:, j], c=omega_c_vals, cmap="viridis", s=30)
    ax.set_xlabel(f"PC{i+1}")
    ax.set_ylabel(f"PC{j+1}")
    ax.set_title(f"PC{i+1} vs PC{j+1}")

plt.colorbar(sc, ax=axes[-1], label="$\\Omega_c$")
fig.suptitle("PCA latent space colored by $\\Omega_c$", fontsize=12)
plt.tight_layout()
plt.show()

## Summary

| Approach | Latent dims | Emulator calls per prediction | Key advantage |
|----------|-------------|-------------------------------|---------------|
| **Direct GP** | N/A | 30 (one per grid point) | Simple, no compression |
| **PCA + GP** | 3 | 3 (one per component) | Fast, exact linear compression |
| **Autoencoder + GP** | 3 | 3 (one per component) | Captures nonlinear structure |

For smooth cosmological functions, PCA is the sweet spot: 3 components capture
>99.99% of variance, the emulator sees a 3-dimensional target instead of 30,
and the decoder guarantees physically coherent curves.

The autoencoder is more powerful but needs more training data and epochs
to outperform PCA on smooth functions.